# Hotel Service Recommendation System (Ranking)

This notebook trains a ranking model for hotel extra-services recommendations using a shared interaction dataset across hotels.

### Problem Statement
Each `booking_id` represents one email with multiple offered products.
- `purchased=1`: product was purchased,
- `purchased=0`: product was shown but not purchased.

Goal: use guest + booking + product features to produce the best product ranking for each email.

### Workflow
1. **Data Loading & Validation**: Clean data and verify real positive/negative labels.
2. **Leak-Free Group Split**: Split by `booking_id`.
3. **Feature Engineering**: Build train-only aggregate features.
4. **Model Training**: Compare Logistic Regression baseline vs. LightGBM LambdaRank.
5. **Evaluation**: Optimize and report ranking metrics (HitRate@K / Recall@K / NDCG@K).
6. **Inference & Export**: Recommend top products and export deployable artifacts.

In [ ]:
import pandas as pd
import numpy as np
import random
import json
import lightgbm as lgb

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, ParameterGrid
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ndcg_score

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

## 1. Data Loading and Data Integrity Check

We load the shared interaction dataset and first validate an important assumption:
- The data already contains both `purchased=1` and `purchased=0` for each email (`booking_id`).
- This means we should treat each `booking_id` as a ranking group and **must not generate synthetic negatives**.

In [ ]:
# Load data from CSV
csv_file_path = 'extraservices_dataset_2026-02-06_09-34-55.csv'

# Load into DataFrame
raw_df = pd.read_csv(csv_file_path)

# Basic preprocessing
# - remove exact duplicate rows
# - fill missing user profile fields
# - sort by booking for deterministic grouped operations
df = (
    raw_df
    .drop_duplicates()
    .assign(
        guest_country_code=lambda x: x['guest_country_code'].fillna('unknown'),
        guest_language=lambda x: x['guest_language'].fillna('unknown')
    )
    .sort_values(['booking_id', 'product_id'])
    .reset_index(drop=True)
)

# Data integrity checks
class_dist = df['purchased'].value_counts().sort_index()
per_booking = df.groupby('booking_id').agg(
    n_items=('product_id', 'count'),
    n_positive=('purchased', 'sum')
)

print(f"Raw rows: {len(raw_df):,}")
print(f"Rows after dedupe: {len(df):,}")
print(f"Bookings: {df['booking_id'].nunique():,}")
print(f"Hotels: {df['hotel_id'].nunique():,}")
print("\nPurchased distribution (0/1):")
print(class_dist)
print(f"\nAvg products shown per email: {per_booking['n_items'].mean():.2f}")
print(f"Avg purchased products per email: {per_booking['n_positive'].mean():.2f}")
print(f"Bookings with at least one positive: {(per_booking['n_positive'] > 0).mean() * 100:.1f}%")

assert set(class_dist.index.tolist()) == {0, 1}, "Dataset must contain both classes 0/1."
assert (per_booking['n_positive'] > 0).all(), "Each booking should have at least one purchased product."

df.head()

## 2. Group Definition (No Synthetic Negative Sampling)

The dataset already contains real negatives (`purchased=0`) and positives (`purchased=1`) for each email.

Therefore:
- `booking_id` is the ranking query/group.
- Rows in one `booking_id` are products shown in the same email.
- The target is to rank purchased products above non-purchased products within that group.

No synthetic negative rows are generated.

In [ ]:
# Keep dataset as-is; interactions are already labeled per booking/email.
full_df = df.copy()

print("Class distribution after cleanup (no synthetic sampling):")
print(full_df['purchased'].value_counts().sort_index())

print("\nQuick sanity check per booking:")
sanity = full_df.groupby('booking_id')['purchased'].agg(['count', 'sum']).head()
print(sanity)

full_df.head()

## 3. Feature Engineering and Leak-Free Group Split

We split by `booking_id` to avoid train/test leakage across products from the same email.

Feature strategy:
- **Numerical**: scaled (`StandardScaler`) for Logistic Regression baseline.
- **Binary**: passthrough.
- **Low-card categorical**: one-hot for Logistic Regression baseline.
- **High-card categorical** (`hotel_id`, `guest_country_code`): ordinal integer encoding for both models.
- **Engineered features**: computed from train statistics only (`price_rel_to_hotel_mean`, `product_popularity_in_hotel`, `guest_avg_lead_days`, `guest_seg_product_cat_rate`).

In [ ]:
# ------------------------
# Group split by booking_id
# ------------------------
features_seed = full_df.copy()
groups = features_seed['booking_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(features_seed, features_seed['purchased'], groups=groups))

train_df = features_seed.iloc[train_idx].copy()
test_df = features_seed.iloc[test_idx].copy()

# ------------------------------------
# Feature engineering from train only
# ------------------------------------
def compute_train_statistics(train_data):
    positive = train_data[train_data['purchased'] == 1]

    hotel_avg_price = train_data.groupby('hotel_id')['product_price_usd'].mean()
    product_popularity = train_data.groupby(['hotel_id', 'product_id'])['purchased'].mean()
    guest_avg_lead = train_data.groupby('guest_id')['booking_lead_days'].mean()
    seg_cat_rate = train_data.groupby(['guest_segment', 'product_category'])['purchased'].mean()

    return {
        'hotel_avg_price': hotel_avg_price,
        'product_popularity': product_popularity,
        'guest_avg_lead': guest_avg_lead,
        'seg_cat_rate': seg_cat_rate,
        'global_avg_price': train_data['product_price_usd'].mean(),
        'global_popularity': train_data['purchased'].mean(),
        'global_guest_lead': train_data['booking_lead_days'].mean(),
        'global_seg_cat_rate': train_data['purchased'].mean(),
    }


def apply_train_statistics(data, stats):
    out = data.copy()

    out['hotel_avg_price'] = out['hotel_id'].map(stats['hotel_avg_price']).fillna(stats['global_avg_price'])
    out['price_rel_to_hotel_mean'] = out['product_price_usd'] / out['hotel_avg_price'].replace(0, stats['global_avg_price'])

    pair_index = pd.MultiIndex.from_frame(out[['hotel_id', 'product_id']])
    out['product_popularity_in_hotel'] = stats['product_popularity'].reindex(pair_index).to_numpy()
    out['product_popularity_in_hotel'] = out['product_popularity_in_hotel'].fillna(stats['global_popularity'])

    out['guest_avg_lead_days'] = out['guest_id'].map(stats['guest_avg_lead']).fillna(stats['global_guest_lead'])

    seg_cat_index = pd.MultiIndex.from_frame(out[['guest_segment', 'product_category']])
    out['guest_seg_product_cat_rate'] = stats['seg_cat_rate'].reindex(seg_cat_index).to_numpy()
    out['guest_seg_product_cat_rate'] = out['guest_seg_product_cat_rate'].fillna(stats['global_seg_cat_rate'])

    return out


stats = compute_train_statistics(train_df)
train_fe = apply_train_statistics(train_df, stats)
test_fe = apply_train_statistics(test_df, stats)

# ---------------------------------------
# High-cardinality category integer maps
# ---------------------------------------
high_card_cols = ['hotel_id', 'guest_country_code']
high_card_maps = {}

for col in high_card_cols:
    train_values = train_fe[col].astype(str)
    categories = sorted(train_values.unique().tolist())
    unknown_token = '__unknown__'
    categories.append(unknown_token)
    mapping = {val: idx for idx, val in enumerate(categories)}
    high_card_maps[col] = mapping

    train_fe[f'{col}_idx'] = train_values.map(mapping).astype(int)
    test_values = test_fe[col].astype(str)
    test_fe[f'{col}_idx'] = test_values.map(lambda x: mapping.get(x, mapping[unknown_token])).astype(int)

# ---------------------------------
# Final feature lists for both models
# ---------------------------------
numerical_cols = [
    'stay_length_days',
    'booking_lead_days',
    'product_price_usd',
    'party_size',
    'product_occupancy_rate',
    'price_rel_to_hotel_mean',
    'product_popularity_in_hotel',
    'guest_avg_lead_days',
    'guest_seg_product_cat_rate'
]

binary_cols = ['is_frequent_guest']
low_card_cat_cols = ['guest_segment', 'season', 'room_type', 'product_category', 'arrival_dow', 'guest_language']
high_card_encoded_cols = ['hotel_id_idx', 'guest_country_code_idx']

rank_features = numerical_cols + binary_cols + low_card_cat_cols + high_card_encoded_cols

# Prepare ranking frames sorted by booking for group arrays
train_rank_df = train_fe.sort_values(['booking_id', 'product_id']).reset_index(drop=True)
test_rank_df = test_fe.sort_values(['booking_id', 'product_id']).reset_index(drop=True)

X_train_rank = train_rank_df[rank_features].copy()
X_test_rank = test_rank_df[rank_features].copy()
y_train = train_rank_df['purchased'].astype(int).to_numpy()
y_test = test_rank_df['purchased'].astype(int).to_numpy()

# Use category dtype for low-card fields in LightGBM
for col in low_card_cat_cols:
    train_cats = pd.Categorical(X_train_rank[col]).categories
    X_train_rank[col] = pd.Categorical(X_train_rank[col], categories=train_cats)
    X_test_rank[col] = pd.Categorical(X_test_rank[col], categories=train_cats)

for col in high_card_encoded_cols:
    X_train_rank[col] = X_train_rank[col].astype('category')
    X_test_rank[col] = X_test_rank[col].astype('category')

train_groups = train_rank_df.groupby('booking_id').size().to_numpy()
test_groups = test_rank_df.groupby('booking_id').size().to_numpy()

# Logistic baseline preprocessing (OHE + scaling)
lr_categorical_cols = low_card_cat_cols + high_card_encoded_cols

lr_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('bin', 'passthrough', binary_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), lr_categorical_cols)
    ]
)

lr_pipeline = Pipeline(steps=[
    ('preprocessor', lr_preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

print(f"Train rows: {len(train_rank_df):,} | Test rows: {len(test_rank_df):,}")
print(f"Train bookings: {train_rank_df['booking_id'].nunique():,} | Test bookings: {test_rank_df['booking_id'].nunique():,}")
print(f"Rank feature count: {len(rank_features)}")

## 4. Model Training (LambdaRank + Logistic Baseline)

In [ ]:
# -----------------
# Train LR baseline
# -----------------
lr_pipeline.fit(X_train_rank, y_train)
lr_test_scores = lr_pipeline.predict_proba(X_test_rank)[:, 1]

# -------------------------
# Train LightGBM LambdaRank
# -------------------------
lgbm_ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)

lgbm_ranker.fit(
    X_train_rank,
    y_train,
    group=train_groups,
    eval_set=[(X_test_rank, y_test)],
    eval_group=[test_groups],
    eval_at=[1, 3, 5],
    categorical_feature=low_card_cat_cols + high_card_encoded_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

lgbm_test_scores = lgbm_ranker.predict(X_test_rank)

print("Training complete for both models.")

## 5. Ranking Evaluation (HitRate@K / Recall@K / NDCG@K)

Global classification metrics can be misleading for ranking tasks.
We evaluate each model within each `booking_id` group and aggregate across bookings.

In [ ]:
def ranking_metrics_by_booking(eval_df, scores, ks=(1, 3, 5)):
    work = eval_df[['booking_id', 'purchased']].copy()
    work['score'] = scores

    metrics = {}

    for k in ks:
        hit_list = []
        recall_list = []

        for _, group in work.groupby('booking_id'):
            topk = group.nlargest(k, 'score')
            positives = group['purchased'].sum()

            hit = int(topk['purchased'].sum() > 0)
            recall = topk['purchased'].sum() / positives if positives > 0 else 0.0

            hit_list.append(hit)
            recall_list.append(recall)

        metrics[f'HitRate@{k}'] = float(np.mean(hit_list))
        metrics[f'Recall@{k}'] = float(np.mean(recall_list))

    # Mean NDCG@K across groups
    for k in ks:
        ndcgs = []
        for _, group in work.groupby('booking_id'):
            y_true = group['purchased'].to_numpy().reshape(1, -1)
            y_score = group['score'].to_numpy().reshape(1, -1)
            ndcgs.append(ndcg_score(y_true, y_score, k=k))
        metrics[f'NDCG@{k}'] = float(np.mean(ndcgs))

    return metrics


lr_metrics = ranking_metrics_by_booking(test_rank_df, lr_test_scores, ks=(1, 3, 5))
lgbm_metrics = ranking_metrics_by_booking(test_rank_df, lgbm_test_scores, ks=(1, 3, 5))

comparison_df = pd.DataFrame([
    {'model': 'LogisticRegression', **lr_metrics},
    {'model': 'LGBMRanker_LambdaRank', **lgbm_metrics}
]).set_index('model')

# Store for final report and export section
final_eval_table = comparison_df.copy()

comparison_df

## 6. Grouped Hyperparameter Tuning (NDCG@5)

This section is production-hardened with two modes:
- `fast`: fewer combinations and 3-fold grouped CV (quick iteration),
- `exhaustive`: full grid and 5-fold grouped CV (best quality, slower).

In [ ]:
# Production tuning knobs
TUNING_MODE = 'fast'  # options: 'fast' or 'exhaustive'

if TUNING_MODE == 'fast':
    param_grid = {
        'num_leaves': [15, 31],
        'learning_rate': [0.05, 0.1],
        'min_child_samples': [10, 20],
    }
    n_splits = 3
    n_estimators_cv = 220
else:
    param_grid = {
        'num_leaves': [15, 31, 63],
        'learning_rate': [0.03, 0.05, 0.1],
        'min_child_samples': [10, 20, 50],
    }
    n_splits = 5
    n_estimators_cv = 300



def prepare_fold_features(train_fold, valid_fold):
    stats_fold = compute_train_statistics(train_fold)
    train_fold_fe = apply_train_statistics(train_fold, stats_fold)
    valid_fold_fe = apply_train_statistics(valid_fold, stats_fold)

    # Encode high-card columns with train-only mapping
    for col in high_card_cols:
        tr_values = train_fold_fe[col].astype(str)
        cats = sorted(tr_values.unique().tolist())
        unknown_token = '__unknown__'
        cats.append(unknown_token)
        mapping = {v: i for i, v in enumerate(cats)}

        train_fold_fe[f'{col}_idx'] = tr_values.map(mapping).astype(int)
        va_values = valid_fold_fe[col].astype(str)
        valid_fold_fe[f'{col}_idx'] = va_values.map(lambda x: mapping.get(x, mapping[unknown_token])).astype(int)

    train_fold_fe = train_fold_fe.sort_values(['booking_id', 'product_id']).reset_index(drop=True)
    valid_fold_fe = valid_fold_fe.sort_values(['booking_id', 'product_id']).reset_index(drop=True)

    X_tr = train_fold_fe[rank_features].copy()
    X_va = valid_fold_fe[rank_features].copy()
    y_tr = train_fold_fe['purchased'].astype(int).to_numpy()
    y_va = valid_fold_fe['purchased'].astype(int).to_numpy()

    for col in low_card_cat_cols:
        tr_cats = pd.Categorical(X_tr[col]).categories
        X_tr[col] = pd.Categorical(X_tr[col], categories=tr_cats)
        X_va[col] = pd.Categorical(X_va[col], categories=tr_cats)

    for col in high_card_encoded_cols:
        X_tr[col] = X_tr[col].astype('category')
        X_va[col] = X_va[col].astype('category')

    g_tr = train_fold_fe.groupby('booking_id').size().to_numpy()
    g_va = valid_fold_fe.groupby('booking_id').size().to_numpy()

    return X_tr, y_tr, g_tr, X_va, y_va, g_va, valid_fold_fe


cv = GroupKFold(n_splits=n_splits)
cv_results = []

for params in ParameterGrid(param_grid):
    fold_scores = []

    for tr_idx, va_idx in cv.split(features_seed, features_seed['purchased'], groups=features_seed['booking_id']):
        train_fold = features_seed.iloc[tr_idx].copy()
        valid_fold = features_seed.iloc[va_idx].copy()

        X_tr, y_tr, g_tr, X_va, y_va, g_va, valid_fold_fe = prepare_fold_features(train_fold, valid_fold)

        fold_model = lgb.LGBMRanker(
            objective='lambdarank',
            metric='ndcg',
            n_estimators=n_estimators_cv,
            num_leaves=params['num_leaves'],
            learning_rate=params['learning_rate'],
            min_child_samples=params['min_child_samples'],
            random_state=42,
        )

        fold_model.fit(
            X_tr,
            y_tr,
            group=g_tr,
            eval_set=[(X_va, y_va)],
            eval_group=[g_va],
            eval_at=[5],
            categorical_feature=low_card_cat_cols + high_card_encoded_cols,
            callbacks=[lgb.early_stopping(25), lgb.log_evaluation(0)],
        )

        va_scores = fold_model.predict(X_va)
        fold_ndcg5 = ranking_metrics_by_booking(valid_fold_fe, va_scores, ks=(5,))['NDCG@5']
        fold_scores.append(fold_ndcg5)

    cv_results.append({
        **params,
        'cv_mean_ndcg5': float(np.mean(fold_scores)),
        'cv_std_ndcg5': float(np.std(fold_scores)),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values('cv_mean_ndcg5', ascending=False).reset_index(drop=True)
best_params = cv_results_df.iloc[0].to_dict()

print(f'Best grouped-CV params (mode={TUNING_MODE}, NDCG@5):')
print(best_params)

# Retrain tuned model on the train split from Section 3 and evaluate on held-out test split
lgbm_ranker_tuned = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=600,
    num_leaves=int(best_params['num_leaves']),
    learning_rate=float(best_params['learning_rate']),
    min_child_samples=int(best_params['min_child_samples']),
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
)

lgbm_ranker_tuned.fit(
    X_train_rank,
    y_train,
    group=train_groups,
    eval_set=[(X_test_rank, y_test)],
    eval_group=[test_groups],
    eval_at=[1, 3, 5],
    categorical_feature=low_card_cat_cols + high_card_encoded_cols,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

tuned_test_scores = lgbm_ranker_tuned.predict(X_test_rank)
tuned_metrics = ranking_metrics_by_booking(test_rank_df, tuned_test_scores, ks=(1, 3, 5))

comparison_df.loc['LGBMRanker_Tuned'] = tuned_metrics
final_eval_table = comparison_df.copy()

cv_results_df.head(10)

## 7. Inference: Recommend Products for a New Guest (Tuned LambdaRank)

This function uses the same feature logic as training:
- hotel product catalog from cleaned interaction data,
- engineered features from train statistics,
- high-cardinality mappings with unknown fallback,
- ranking by tuned LambdaRank score.

In [ ]:
def recommend_for_new_guest(guest_dict, hotel_id, model, source_df, top_k=5):
    """
    Rank hotel products for a new guest context using tuned LambdaRank model.
    """
    product_cols = ['hotel_id', 'product_id', 'product_category', 'product_price_usd', 'product_occupancy_rate']
    hotel_catalog = source_df[source_df['hotel_id'] == hotel_id][product_cols].drop_duplicates()

    if hotel_catalog.empty:
        return pd.DataFrame(columns=['product_id', 'product_category', 'product_price_usd', 'rank_score'])

    defaults = {
        'guest_id': '__new_guest__',
        'guest_segment': 'leisure',
        'is_frequent_guest': 0,
        'stay_length_days': 2,
        'arrival_dow': 1,
        'season': 'mid',
        'room_type': 'STD',
        'booking_lead_days': 14,
        'guest_country_code': 'unknown',
        'guest_language': 'unknown',
        'party_size': 2,
    }

    merged_profile = {**defaults, **guest_dict}

    candidates = hotel_catalog.copy()
    for key, value in merged_profile.items():
        candidates[key] = value

    candidates['booking_id'] = -1
    candidates['purchased'] = 0

    # Apply same train-derived feature transformations
    candidates_fe = apply_train_statistics(candidates, stats)

    for col in high_card_cols:
        mapping = high_card_maps[col]
        unknown_idx = mapping['__unknown__']
        candidates_fe[f'{col}_idx'] = candidates_fe[col].astype(str).map(lambda x: mapping.get(x, unknown_idx)).astype(int)

    X_candidates = candidates_fe[rank_features].copy()

    # Match categorical dtypes used during model training
    for col in low_card_cat_cols:
        X_candidates[col] = pd.Categorical(X_candidates[col], categories=X_train_rank[col].cat.categories)

    for col in high_card_encoded_cols:
        X_candidates[col] = X_candidates[col].astype('category')

    candidates_fe['rank_score'] = model.predict(X_candidates)

    ranked = candidates_fe.sort_values('rank_score', ascending=False).head(top_k)
    return ranked[['product_id', 'product_category', 'product_price_usd', 'rank_score']]


# --- DEMO ---
new_guest_profile = {
    'guest_id': 'demo_guest_001',
    'guest_segment': 'business',
    'is_frequent_guest': 1,
    'stay_length_days': 2,
    'arrival_dow': 1,
    'season': 'peak',
    'room_type': 'DLX',
    'booking_lead_days': 7,
    'guest_country_code': 'de',
    'guest_language': 'de',
    'party_size': 1,
}

print("Top recommendations for demo guest:")
recommend_for_new_guest(
    guest_dict=new_guest_profile,
    hotel_id=1028,
    model=lgbm_ranker_tuned,
    source_df=full_df,
    top_k=5,
)

## 8. Model Export (Ranker Primary + LR PHP Fallback)

We export:
- tuned LightGBM ranker as `lgbm_ranker_model.txt`,
- metadata required for feature construction as `lgbm_ranker_metadata.json`,
- Logistic Regression JSON parameters as PHP fallback (`lr_model_parameters.json`).

In [ ]:
# ---------------------------
# Export tuned LightGBM model
# ---------------------------
lgbm_ranker_tuned.booster_.save_model('lgbm_ranker_model.txt')

ranker_metadata = {
    'metadata_version': '1.0',
    'rank_features': rank_features,
    'numerical_cols': numerical_cols,
    'binary_cols': binary_cols,
    'low_card_cat_cols': low_card_cat_cols,
    'low_card_categories': {
        col: [str(v) for v in X_train_rank[col].cat.categories.tolist()]
        for col in low_card_cat_cols
    },
    'high_card_cols': high_card_cols,
    'high_card_encoded_cols': high_card_encoded_cols,
    'high_card_maps': high_card_maps,
    'train_stats': {
        'hotel_avg_price': {str(k): float(v) for k, v in stats['hotel_avg_price'].to_dict().items()},
        'product_popularity': {f"{k[0]}::{k[1]}": float(v) for k, v in stats['product_popularity'].to_dict().items()},
        'guest_avg_lead': {str(k): float(v) for k, v in stats['guest_avg_lead'].to_dict().items()},
        'seg_cat_rate': {f"{k[0]}::{k[1]}": float(v) for k, v in stats['seg_cat_rate'].to_dict().items()},
        'global_avg_price': float(stats['global_avg_price']),
        'global_popularity': float(stats['global_popularity']),
        'global_guest_lead': float(stats['global_guest_lead']),
        'global_seg_cat_rate': float(stats['global_seg_cat_rate']),
    }
}

with open('lgbm_ranker_metadata.json', 'w') as f:
    json.dump(ranker_metadata, f, indent=2)

# ----------------------------------------
# Export LR fallback for pure PHP scoring
# ----------------------------------------
lr_num_scaler = lr_pipeline.named_steps['preprocessor'].named_transformers_['num']
lr_ohe = lr_pipeline.named_steps['preprocessor'].named_transformers_['cat']
lr_model = lr_pipeline.named_steps['classifier']

lr_export = {
    'scaler': {
        'numerical_features': numerical_cols,
        'mean': lr_num_scaler.mean_.tolist(),
        'scale': lr_num_scaler.scale_.tolist(),
    },
    'ohe': {
        'categorical_features': lr_categorical_cols,
        'categories': [cats.tolist() for cats in lr_ohe.categories_],
    },
    'model': {
        'coefficients': lr_model.coef_[0].tolist(),
        'intercept': float(lr_model.intercept_[0]),
    }
}

with open('lr_model_parameters.json', 'w') as f:
    json.dump(lr_export, f, indent=2)

print('Saved: lgbm_ranker_model.txt, lgbm_ranker_metadata.json, lr_model_parameters.json')

## 9. Final Review: What Was Fixed and Why It Improves Accuracy

Implemented changes vs original notebook:
- Removed synthetic negative sampling (dataset already contains real negatives per email).
- Switched to leak-free split by `booking_id`.
- Optimized for ranking quality (HitRate@K / Recall@K / NDCG@K), not global ROC-AUC.
- Added LambdaRank model with grouped CV tuning.
- Kept Logistic Regression baseline as deployment fallback.

The table below reports final held-out ranking performance and relative lift of tuned LambdaRank over Logistic Regression.

In [ ]:
print('Final held-out ranking metrics:')
print(final_eval_table)

if 'LGBMRanker_Tuned' in final_eval_table.index and 'LogisticRegression' in final_eval_table.index:
    lr_row = final_eval_table.loc['LogisticRegression']
    tuned_row = final_eval_table.loc['LGBMRanker_Tuned']

    lift = pd.DataFrame({
        'LR_baseline': lr_row,
        'LGBM_tuned': tuned_row,
        'abs_delta': tuned_row - lr_row,
        'relative_delta_%': ((tuned_row - lr_row) / lr_row.replace(0, np.nan)) * 100,
    })

    print('\nLift of tuned LambdaRank over LR baseline:')
    print(lift)
else:
    print('Run training/tuning cells first to produce final comparison.')